
# Notebook: Digital Twin v1
Phase: 3 — Temporal Modeling & Body States

Objectif:
- Transformer les scores statiques (Phase 2) en états vivants
- Ajouter la dimension temporelle (timeline)
- Définir des états corporels simples
- Calculer baseline et déviations individuelles

Scores couverts:
- Body Age
- Work Load
- Body Toxins


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/PerenAI_3scores.csv")

df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values(["user_id", "datetime"])



df.head()


,user_id,datetime,sex,age,height_cm,weight_kg,body_age,work_load,body_toxin
0,U01,2025-10-18 08:12:34,female,32,165,58,32.0,10.0,2.0
1,U01,2025-11-02 07:58:21,female,32,165,59,32.0,10.0,2.0
2,U01,2025-11-15 08:20:41,female,32,165,58,32.0,10.0,2.0
3,U02,2025-10-20 10:45:12,male,38,178,75,40.0,40.0,4.0
4,U02,2025-11-04 09:40:08,male,38,178,76,39.0,30.0,4.0


# Body age - Digital Twin

In [2]:
df["delta_body_age"] = df["body_age"] - df["age"]

# Etats Body age
#On traduit un chiffre en état biologique lisible.
def body_age_state(delta):
    if delta <= -2:
        return "younger_than_chrono"
    elif -1 <= delta <= 1:
        return "aligned_with_chrono"
    else:
        return "accelerated_aging"

df["body_age_state"] = df["delta_body_age"].apply(body_age_state)



In [3]:
#Change vs last period
df["body_age_change"] = (
    df.groupby("user_id")["body_age"]
      .diff()
      .fillna(0)
)


# WORK LOAD — Digital Twin

In [4]:
def workload_state(score):
    if score <= 0:
        return "low_load"
    elif score <= 30:
        return "moderate_load"
    else:
        return "high_load"

df["workload_state"] = df["work_load"].apply(workload_state)


In [5]:
df["work_load_change"] = (
    df.groupby("user_id")["work_load"]
      .diff()
      .fillna(0)
)


# BODY TOXINS — Digital Twin

In [6]:
def toxin_state(score):
    if score <= 0:
        return "low_toxic_load"
    elif score <= 2:
        return "moderate_toxic_load"
    else:
        return "high_toxic_load"

df["body_toxins_state"] = df["body_toxin"].apply(toxin_state)


In [7]:
df["body_toxin_change"] = (
    df.groupby("user_id")["body_toxin"]
      .diff()
      .fillna(0)
)


In [8]:
df["global_body_state"] = (
    df["body_age_state"] + " | "
    + df["workload_state"] + " | "
    + df["body_toxins_state"]
)


In [9]:
final_cols = [
    "user_id",
    "datetime",
    "sex",
    "age",
    "height_cm",
    "weight_kg",

    "body_age",
    "body_age_change",
    "body_age_state",

    "work_load",
    "work_load_change",
    "workload_state",

    "body_toxin",
    "body_toxin_change",
    "body_toxins_state",

]

df_final = df[final_cols].copy()

df_final.head()


,user_id,datetime,sex,age,height_cm,weight_kg,body_age,body_age_change,body_age_state,work_load,work_load_change,workload_state,body_toxin,body_toxin_change,body_toxins_state
0,U01,2025-10-18 08:12:34,female,32,165,58,32.0,0.0,aligned_with_chrono,10.0,0.0,moderate_load,2.0,0.0,moderate_toxic_load
1,U01,2025-11-02 07:58:21,female,32,165,59,32.0,0.0,aligned_with_chrono,10.0,0.0,moderate_load,2.0,0.0,moderate_toxic_load
2,U01,2025-11-15 08:20:41,female,32,165,58,32.0,0.0,aligned_with_chrono,10.0,0.0,moderate_load,2.0,0.0,moderate_toxic_load
3,U02,2025-10-20 10:45:12,male,38,178,75,40.0,0.0,accelerated_aging,40.0,0.0,high_load,4.0,0.0,high_toxic_load
4,U02,2025-11-04 09:40:08,male,38,178,76,39.0,-1.0,aligned_with_chrono,30.0,-10.0,moderate_load,4.0,0.0,high_toxic_load


In [10]:
df_final.to_csv("../data/processed/PerenAI_digital_twin_v1.csv", index=False)
